# Radar Signal Visualisation

This notebook examines representative complex-valued radar segments from drones and birds.

The objectives are to:

1. Select representative training samples from each target category.
2. Reshape each 1,280-element segment into five range cells and 256 slow-time samples.
3. Visualise signal magnitude and phase.
4. Apply Doppler processing along the slow-time dimension.
5. Compare the resulting radar signatures across target categories.

## 1. Dataset Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

FILE_PATH = Path("../data/raw/data_SAAB_SIRS_77GHz_FMCW.npy")

if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {FILE_PATH.resolve()}"
    )

data = np.load(FILE_PATH, allow_pickle=True)
print("Dataset shape:", data.shape)
print("Dataset dtype:", data.dtype)

## 2. Target-Label Utilities

In [ ]:
DRONE_LABELS = {"D1", "D2", "D3", "D4", "D5", "D6"}

BIRD_LABELS = {
    "seagull",
    "pigeon",
    "raven",
    "black-headed gull",
    "seagull and black-headed gull",
    "heron"
}

def extract_label(value):
    """Extract a clean string label from a dataset label field."""
    label_array = np.asarray(value).reshape(-1)

    if label_array.size == 0:
        return "unknown"

    return str(label_array[0]).strip()


def map_target_group(label):
    """Map an original target label to a broader target group."""
    if label in DRONE_LABELS:
        return "drone"

    if label in BIRD_LABELS:
        return "bird"

    if label in {"human_walk", "human_run"}:
        return "human"

    if label == "CR":
        return "corner_reflector"

    return "unknown"

## 3. Representative Radar Segment Selection

A representative sample is selected using only the official training partition (`split == 1`) and observations that are not truncated at the field-of-view boundary (`edge_flag == 0`). Within the first suitable session, the selected segment is the one recorded closest to that session's median valid range.

This deterministic rule supports reproducible visualisation. It does not guarantee that the selected observation is statistically representative of the complete target class.


In [ ]:
def select_representative_segment(dataset, target_label):
    """
    Select a non-edge training segment close to the median range
    of the first suitable session for the requested target.
    """
    for session_id, row in enumerate(dataset):
        label = extract_label(row[0])

        if label != target_label:
            continue

        segments = np.asarray(row[1])
        ranges = np.asarray(row[2]).reshape(-1)
        times = np.asarray(row[3]).reshape(-1)
        splits = np.asarray(row[4]).reshape(-1)
        edge_flags = np.asarray(row[5]).reshape(-1)

        valid_indices = np.where(
            (splits == 1) & (edge_flags == 0)
        )[0]

        if valid_indices.size == 0:
            continue

        valid_ranges = ranges[valid_indices]
        median_range = np.median(valid_ranges)

        relative_index = np.argmin(
            np.abs(valid_ranges - median_range)
        )
        segment_index = valid_indices[relative_index]

        segment = segments[:, segment_index].reshape(5, 256)

        return {
            "session_id": session_id,
            "segment_id": int(segment_index),
            "label": label,
            "target_group": map_target_group(label),
            "range_m": float(ranges[segment_index]),
            "time_s": float(times[segment_index]),
            "segment": segment
        }

    raise ValueError(
        f"No suitable training segment found for target {target_label}"
    )

In [ ]:
example = select_representative_segment(data, "D1")

print("Target label:", example["label"])
print("Target group:", example["target_group"])
print("Session ID:", example["session_id"])
print("Segment ID:", example["segment_id"])
print("Target range (m):", example["range_m"])
print("Timestamp (s):", example["time_s"])
print("Segment shape:", example["segment"].shape)
print("Complex-valued:", np.iscomplexobj(example["segment"]))

### Selected D1 Segment

The selected example belongs to `D1`, the DJI Matrice 200 V drone. It comes from session 0 and segment 2, was recorded at approximately **62.26 m** and **0.240 s**, and has shape **(5, 256)**.

The complex-valued representation preserves magnitude and phase information required for Doppler processing. The sample belongs to the official training set and is not marked as a field-of-view edge observation.

## 4. Raw Complex-Signal Visualisation

The magnitude in decibels is calculated as

$$M_{\mathrm{dB}} = 20\log_{10}(|x|+\epsilon),$$

while the phase is $\phi=\arg(x)$. Magnitude describes echo strength, whereas phase changes contain information related to motion.

In [ ]:
segment = example["segment"]

magnitude_db = 20 * np.log10(np.abs(segment) + 1e-12)
phase_rad = np.angle(segment)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)

magnitude_image = axes[0].imshow(
    magnitude_db,
    aspect="auto",
    origin="lower",
    cmap="viridis"
)
axes[0].set_title(
    f"Raw Signal Magnitude — {example['label']} "
    f"at {example['range_m']:.1f} m"
)
axes[0].set_xlabel("Slow-time / azimuth sample")
axes[0].set_ylabel("Range cell")
fig.colorbar(
    magnitude_image,
    ax=axes[0],
    label="Magnitude (dB)"
)

phase_image = axes[1].imshow(
    phase_rad,
    aspect="auto",
    origin="lower",
    cmap="twilight",
    vmin=-np.pi,
    vmax=np.pi
)
axes[1].set_title("Raw Signal Phase")
axes[1].set_xlabel("Slow-time / azimuth sample")
axes[1].set_ylabel("Range cell")
fig.colorbar(
    phase_image,
    ax=axes[1],
    label="Phase (radians)"
)

plt.show()

### Raw Signal Observations

The strongest return is concentrated mainly in the central range cell 2 and neighbouring cells 1 and 3. This is consistent with the dataset construction, in which the detected target is centred on the third range cell.

Magnitude increases near the centre of the 256-sample interval as the mechanically scanning antenna approaches and crosses the target direction. The outer range cells contain weaker, less structured measurements that are more strongly influenced by noise and clutter.

The central range cells also show structured phase variation, whereas phase in the weaker outer cells appears more irregular. The raw representation nevertheless combines antenna scanning, target position, motion, clutter, and noise. Doppler processing is therefore required to inspect velocity-dependent energy.

## 5. Range–Doppler Processing

Following the processing interval reported with the dataset, central indices 54–203 are retained. A Hann window reduces spectral leakage before an FFT is applied along slow time:

$$D(r,f)=\left|\mathcal{F}\{w[n]x(r,n)\}\right|.$$

Doppler frequency is converted to approximate radial velocity using

$$v=\frac{\lambda f_D}{2}.$$

Every patch is expressed relative to its own maximum, so 0 dB denotes the strongest value in that patch.

In [ ]:
PRF_HZ = 17_000
CARRIER_FREQUENCY_HZ = 77e9
SPEED_OF_LIGHT_M_S = 299_792_458

WAVELENGTH_M = SPEED_OF_LIGHT_M_S / CARRIER_FREQUENCY_HZ

AZIMUTH_START = 54
AZIMUTH_END = 204

def compute_range_doppler(segment):
    """
    Convert a (5, 256) complex radar segment into a
    normalised range-Doppler magnitude representation.
    """
    central_samples = segment[:, AZIMUTH_START:AZIMUTH_END]

    window = np.hanning(central_samples.shape[1])
    windowed_signal = central_samples * window[np.newaxis, :]

    doppler_complex = np.fft.fftshift(
        np.fft.fft(windowed_signal, axis=1),
        axes=1
    )

    doppler_magnitude = np.abs(doppler_complex)
    doppler_db = 20 * np.log10(doppler_magnitude + 1e-12)

    # Normalise each representation relative to its maximum.
    doppler_db = doppler_db - np.max(doppler_db)

    doppler_frequencies = np.fft.fftshift(
        np.fft.fftfreq(
            central_samples.shape[1],
            d=1 / PRF_HZ
        )
    )

    velocity_axis = (
        doppler_frequencies * WAVELENGTH_M / 2
    )

    return doppler_db, velocity_axis

In [ ]:
range_doppler_db, velocity_axis = compute_range_doppler(segment)

print("Range-Doppler shape:", range_doppler_db.shape)
print(
    "Approximate velocity interval:",
    f"{velocity_axis.min():.2f} to "
    f"{velocity_axis.max():.2f} m/s"
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.imshow(
    range_doppler_db,
    aspect="auto",
    origin="lower",
    cmap="magma",
    extent=[
        velocity_axis[0],
        velocity_axis[-1],
        -0.5,
        4.5
    ],
    vmin=-60,
    vmax=0
)

plt.colorbar(label="Relative magnitude (dB)")
plt.xlabel("Radial velocity (m/s)")
plt.ylabel("Range cell")
plt.yticks(range(5))
plt.title(
    f"Range–Doppler Representation — {example['label']} "
    f"at {example['range_m']:.1f} m"
)
plt.show()

### D1 Range–Doppler Observations

The D1 patch has shape **(5, 150)** and contains a dominant response close to zero radial velocity, concentrated mainly in range cell 2 with energy extending into cells 1 and 3. Weaker energy spreads into neighbouring positive and negative velocity bins. Possible contributors include drone-body motion, propeller motion, antenna scanning, spectral leakage, noise, and clutter.

This representation covers only approximately 15 ms. It is therefore a short range–Doppler patch, not a conventional long-duration micro-Doppler spectrogram. Temporal sequences of consecutive patches may later be needed to represent complete periodic rotor or wing-flapping patterns.

## 6. Qualitative Comparison Across Target Categories

One valid training segment is selected for each drone model and bird category. All examples use identical processing and a shared display range of −60 to 0 dB. Because every patch is normalised independently, the comparison emphasises spectral shape rather than absolute echo strength.

In [ ]:
TARGET_LABELS = [
    "D1", "D2", "D3", "D4", "D5", "D6",
    "seagull",
    "black-headed gull",
    "seagull and black-headed gull",
    "heron",
    "pigeon",
    "raven"
]

representative_examples = {
    label: select_representative_segment(data, label)
    for label in TARGET_LABELS
}

fig, axes = plt.subplots(
    4,
    3,
    figsize=(16, 14),
    constrained_layout=True
)

for axis, label in zip(axes.flat, TARGET_LABELS):
    selected = representative_examples[label]

    doppler_db, velocity = compute_range_doppler(
        selected["segment"]
    )

    image = axis.imshow(
        doppler_db,
        aspect="auto",
        origin="lower",
        cmap="magma",
        extent=[
            velocity[0],
            velocity[-1],
            -0.5,
            4.5
        ],
        vmin=-60,
        vmax=0
    )

    axis.set_title(
        f"{label} — {selected['range_m']:.1f} m"
    )
    axis.set_xlabel("Radial velocity (m/s)")
    axis.set_ylabel("Range cell")
    axis.set_yticks(range(5))

fig.colorbar(
    image,
    ax=axes,
    label="Relative magnitude (dB)",
    shrink=0.8
)

plt.show()

### Qualitative Comparison Results

Most examples contain a dominant component near zero radial velocity, but their bandwidth, symmetry, range-cell distribution, and relative background level vary.

- `D1`, `D3`, `D4`, and `D6` show comparatively concentrated central components.
- `D2` and `D5` exhibit broader distributions across neighbouring velocity bins.
- The gull examples contain narrow dominant components with weaker surrounding energy.
- The selected pigeon and raven examples have strong responses at negative non-zero velocities.
- The selected heron patch has a comparatively high relative background and lower contrast.

These observations suggest that the patches may contain useful classification information, but they do not establish drone–bird separability. Each panel represents only one sample, and targets were measured at different ranges, velocities, sessions, and signal-to-noise conditions.

## 7. Conclusion and Next Steps

The complex radar segments were successfully reshaped to **(5, 256)** and transformed into short range–Doppler patches of **(5, 150)**. Target energy is generally centred in the middle range cells, while Doppler-energy structure varies across categories. This provides a plausible foundation for machine-learning classification.

The visual differences cannot yet be attributed exclusively to target type. They may also reflect target range, radial velocity, session conditions, signal-to-noise ratio, antenna position, flight behaviour, and independent per-patch normalisation.

The next stage will therefore:

1. Process multiple samples from multiple sessions.
2. Quantify Doppler centroid, bandwidth, energy concentration, and range distribution.
3. Examine the influence of target range and session identity.
4. Define a reproducible preprocessing and normalisation pipeline.
5. Build an individual-patch baseline classifier.
6. Later compare it with a temporal model built from consecutive patches.

All final classification claims will be based on unseen real radar measurements. Synthetic data will be introduced only into the training partition during the later limited-data augmentation experiments.